# JSON to Excel Export (Template-First)

This notebook exports one workbook per mold family from `mold_data.json`.

Rules implemented:
- Always load the latest template workbook.
- Follow template sheet/table/range names as source of truth.
- Keep null `vendorShortName` as blank.
- Ignore non-numeric mold-size keys in sizing maps.

In [4]:

import json
import re
from pathlib import Path

import pythoncom
import win32com.client as win32


def to_stripped(value):
    if value is None:
        return None
    text = str(value).strip()
    return text if text else None


def as_number_or_none(value):
    if value is None:
        return None
    try:
        return float(value)
    except Exception:
        return None


def is_numeric_size_key(value):
    if value is None:
        return False
    try:
        float(str(value).strip())
        return True
    except Exception:
        return False


INVALID_SHEET_CHARS = r"[\[\]\:\*\?\/\\]"


def safe_sheet_name(name):
    name = "Sheet" if name is None else str(name)
    name = re.sub(INVALID_SHEET_CHARS, "-", name).strip()
    if len(name) > 31:
        name = name[:31]
    return name or "Sheet"


def mold_size_to_row(size_value):
    size_f = float(size_value)
    step = (size_f - 1.0) / 0.5
    rounded = round(step)
    if abs(step - rounded) > 1e-9:
        return None
    row = 8 + int(rounded)
    if row < 8 or row > 42:
        return None
    return row

In [ ]:
XL_CALC_MANUAL = -4135
XL_CALC_AUTO = -4105

def excel_turbo_on(app):
    app.Visible = False
    app.DisplayAlerts = False
    app.ScreenUpdating = False
    app.EnableEvents = False
    app.AskToUpdateLinks = False
    app.Calculation = XL_CALC_MANUAL

def excel_turbo_off(app):
    app.Calculation = XL_CALC_AUTO
    app.EnableEvents = True
    app.ScreenUpdating = True
    app.DisplayAlerts = True

In [5]:
def _normalize_table_values(values):
    if values is None:
        return []
    if not isinstance(values, tuple):
        return [[values]]
    if len(values) == 0:
        return []
    first = values[0]
    if isinstance(first, tuple):
        return [list(r) for r in values]
    return [list(values)]


def read_listobject_records(ws, table_name):
    lo = ws.ListObjects(table_name)
    rows = _normalize_table_values(lo.Range.Value)
    if not rows:
        return []
    headers = rows[0]
    out = []
    for row in rows[1:]:
        out.append({headers[i]: row[i] if i < len(row) else None for i in range(len(headers))})
    return out


def build_lookup_maps(ws_master):
    vendor_rows = read_listobject_records(ws_master, "_dimVendor")
    factory_rows = read_listobject_records(ws_master, "_dimFactory")

    vendor_short_by_id = {}
    for row in vendor_rows:
        vendor_id = to_stripped(row.get("Vendor ID"))
        vendor_short = to_stripped(row.get("Vendor Short Name"))
        if vendor_id and vendor_short:
            vendor_short_by_id[vendor_id] = vendor_short

    factory_name_by_number = {}
    for row in factory_rows:
        number = row.get("Factory Number")
        name = to_stripped(row.get("Factory Name"))
        if number is None or not name:
            continue
        try:
            key = str(int(float(number)))
        except Exception:
            key = str(number).strip()
        factory_name_by_number[key] = name

    return vendor_short_by_id, factory_name_by_number


def iter_component_records(fam):
    components = fam.get("components") or {}
    for sole_type, comp_dict in components.items():
        for comp_code, comp in (comp_dict or {}).items():
            molds = comp.get("molds") or []
            for mold in molds:
                yield sole_type, comp_code, comp, mold


def collect_summary_rows(fam):
    rows = []
    for _sole_type, comp_code, _comp, mold in iter_component_records(fam):
        asset = mold.get("asset") or {}
        cap = mold.get("capacity") or {}

        comment = to_stripped(mold.get("comments"))
        remark = to_stripped(mold.get("remark"))
        if comment and remark and comment != remark:
            comment_out = f"{comment} | {remark}"
        else:
            comment_out = comment or remark

        rows.append({
            "component_code": comp_code,
            "vendor_short_name": to_stripped(mold.get("vendorShortName")) or "",
            "ownership": to_stripped(asset.get("ownership")),
            "condition": to_stripped(asset.get("condition")),
            "mold_shop": to_stripped(mold.get("moldShopName")) or to_stripped(mold.get("moldShopCode")),
            "init_season": to_stripped(mold.get("initSeason")),
            "daily_output": as_number_or_none(cap.get("dailyOutput")),
            "mold_init_cost": as_number_or_none(asset.get("moldCost")),
            "comments": comment_out,
        })
    return rows

In [ ]:
def write_range(ws, r1, c1, data_2d):
    r2 = r1 + len(data_2d) - 1
    c2 = c1 + len(data_2d[0]) - 1
    ws.Range(ws.Cells(r1, c1), ws.Cells(r2, c2)).Value = data_2d

def write_summary(ws_summary, family_code, fam, warnings):
    input_start_row = 7
    input_end_row = 31

    ws_summary.Cells(1, 1).Value = f"Mold Family: {family_code}"

    brands = fam.get("brands") or []
    style_brands = [to_stripped(s.get("brand")) for s in (fam.get("stylesUsingThisFamily") or []) if isinstance(s, dict)]
    all_brands = [b for b in brands if to_stripped(b)] + [b for b in style_brands if b]
    unique_brands = []
    for b in all_brands:
        if b not in unique_brands:
            unique_brands.append(b)
    ws_summary.Cells(2, 2).Value = ", ".join(unique_brands) if unique_brands else None

    for r in range(input_start_row, input_end_row + 1):
        for c in [1, 4, 8, 9, 10, 11, 12, 13, 14]:
            ws_summary.Cells(r, c).Value = None

    rows = collect_summary_rows(fam)
    capacity = input_end_row - input_start_row + 1
    if len(rows) > capacity:
        warnings.append(f"{family_code}: Summary rows truncated from {len(rows)} to {capacity}.")
        rows = rows[:capacity]

    for i, row_data in enumerate(rows):
        r = input_start_row + i
        ws_summary.Cells(r, 1).Value = row_data["component_code"]
        ws_summary.Cells(r, 4).Value = row_data["vendor_short_name"] or None
        ws_summary.Cells(r, 8).Value = row_data["ownership"]
        ws_summary.Cells(r, 9).Value = row_data["condition"]
        ws_summary.Cells(r, 10).Value = row_data["mold_shop"]
        ws_summary.Cells(r, 11).Value = row_data["init_season"]
        ws_summary.Cells(r, 12).Value = row_data["daily_output"]
        ws_summary.Cells(r, 13).Value = row_data["mold_init_cost"]
        ws_summary.Cells(r, 14).Value = row_data["comments"]

def write_sourcing_rules(ws_src, fam, vendor_short_by_id, factory_name_by_number, warnings, family_code):
    start_row = 5
    end_row = 29
    capacity = end_row - start_row + 1

    for r in range(start_row, end_row + 1):
        ws_src.Cells(r, 1).Value = None
        ws_src.Cells(r, 4).Value = None
        ws_src.Cells(r, 6).Value = None
        ws_src.Cells(r, 10).Value = None

    rules = fam.get("sourcingRules") or []
    if len(rules) > capacity:
        warnings.append(f"{family_code}: Sourcing Rules truncated from {len(rules)} to {capacity}.")
        rules = rules[:capacity]

    for i, rule in enumerate(rules):
        r = start_row + i
        factory_number = rule.get("factoryNumber")
        if factory_number is None:
            factory_name = None
        else:
            try:
                factory_key = str(int(float(factory_number)))
            except Exception:
                factory_key = str(factory_number).strip()
            factory_name = factory_name_by_number.get(factory_key)

        vendor_id = to_stripped(rule.get("vendorId"))
        vendor_short = vendor_short_by_id.get(vendor_id) if vendor_id else None

        ws_src.Cells(r, 1).Value = factory_name
        ws_src.Cells(r, 4).Value = to_stripped(rule.get("solePart"))
        ws_src.Cells(r, 6).Value = vendor_short

    style_names = []
    for s in (fam.get("stylesUsingThisFamily") or []):
        if isinstance(s, dict):
            style_name = to_stripped(s.get("styleName")) or to_stripped(s.get("styleCode"))
        else:
            style_name = to_stripped(s)
        if style_name and style_name not in style_names:
            style_names.append(style_name)

    if len(style_names) > capacity:
        warnings.append(f"{family_code}: Style list truncated from {len(style_names)} to {capacity}.")
        style_names = style_names[:capacity]

    for i, style_name in enumerate(style_names):
        ws_src.Cells(start_row + i, 10).Value = style_name


def write_component_sheet(ws_inv, family_code, sole_type, comp_code, comp, sizing_block, warnings):
    # headers (few calls OK)
    ws_inv.Cells(1, 1).Value = f"Mold Inventory - {comp_code}"
    ws_inv.Cells(2, 1).Value = f"SoleType: {sole_type}"
    ws_inv.Cells(3, 1).Value = f"Component Name: {to_stripped(comp.get('displayName')) or comp_code}"
    ws_inv.Cells(4, 1).Value = f"Compound: {to_stripped(comp.get('compound')) or ''}"

    # clear B8:H42 in one call
    ws_inv.Range(ws_inv.Cells(8, 2), ws_inv.Cells(42, 8)).Value = None

    molds = comp.get("molds") or []

    # vendor columns (max 6)
    vendor_order = []
    for mold in molds:
        v = to_stripped(mold.get("vendorShortName")) or ""
        if v not in vendor_order:
            vendor_order.append(v)

    max_vendor_cols = 6
    if len(vendor_order) > max_vendor_cols:
        warnings.append(f"{family_code}/{comp_code}: Vendor columns truncated from {len(vendor_order)} to {max_vendor_cols}.")
        vendor_order = vendor_order[:max_vendor_cols]

    vendor_headers = [(vendor_order[i] if i < len(vendor_order) and vendor_order[i] else None) for i in range(max_vendor_cols)]
    vendor_to_idx = {v: i for i, v in enumerate(vendor_order) if v}

    # write vendor headers C7:H7 in ONE call
    write_range(ws_inv, 7, 3, [vendor_headers])

    # build shoe sizes column (B8:B42) in python
    shoe_sizes_col = [None] * 35  # rows 8..42

    sizing_map = (sizing_block or {}).get("moldSizeToShoeSizes") or {}
    for mold_size, shoe_sizes in sizing_map.items():
        if not is_numeric_size_key(mold_size):
            continue
        row = mold_size_to_row(float(str(mold_size).strip()))
        if row is None:
            continue
        idx = row - 8
        if shoe_sizes is None:
            shoe_sizes_col[idx] = None
        elif isinstance(shoe_sizes, list):
            shoe_sizes_col[idx] = ", ".join(str(x) for x in shoe_sizes)
        else:
            shoe_sizes_col[idx] = str(shoe_sizes)

    # build qty matrix (C8:H42) in python
    qty = [[None] * 6 for _ in range(35)]
    for mold in molds:
        v = to_stripped(mold.get("vendorShortName")) or ""
        if v not in vendor_to_idx:
            continue
        col_idx = vendor_to_idx[v]

        qty_map = ((mold.get("inventory") or {}).get("qtyByMoldSize")) or {}
        for mold_size, q in qty_map.items():
            if not is_numeric_size_key(mold_size):
                continue
            row = mold_size_to_row(float(str(mold_size).strip()))
            if row is None:
                continue
            qn = as_number_or_none(q)
            if qn is None:
                continue
            r_idx = row - 8
            cur = qty[r_idx][col_idx] or 0.0
            qty[r_idx][col_idx] = cur + qn

    # write shoe sizes + qty in TWO calls
    write_range(ws_inv, 8, 2, [[x] for x in shoe_sizes_col])  # B8:B42
    write_range(ws_inv, 8, 3, qty)                             # C8:H42


def export_one_family(template_path, family_code, fam, output_dir, app):
    warnings = []
    template_abs = str(Path(template_path).resolve())
    out_path = Path(output_dir).resolve() / f"{family_code}.xlsx"
    out_path.parent.mkdir(parents=True, exist_ok=True)

    wb = app.Workbooks.Open(template_abs)
    try:
        ws_summary = wb.Worksheets("Summary")
        ws_inv_template = wb.Worksheets("MoldInv_{Component}")
        ws_src = wb.Worksheets("Sourcing Rules")
        ws_master = wb.Worksheets("_Master_Ref")

        vendor_short_by_id, factory_name_by_number = build_lookup_maps(ws_master)

        write_summary(ws_summary, family_code, fam, warnings)
        write_sourcing_rules(ws_src, fam, vendor_short_by_id, factory_name_by_number, warnings, family_code)

        components = fam.get("components") or {}
        sizing_all = fam.get("sizingRulesByComponent") or {}

        created_names = set()
        for sole_type, comp_dict in components.items():
            for comp_code, comp in (comp_dict or {}).items():
                ws_inv_template.Copy(After=wb.Worksheets(wb.Worksheets.Count))
                new_ws = wb.Worksheets(wb.Worksheets.Count)

                target_name = safe_sheet_name(f"MoldInv_{comp_code}")
                base_name = target_name
                suffix = 1
                existing = {sh.Name for sh in wb.Worksheets}

                while target_name in existing:
                    suffix += 1
                    target_name = safe_sheet_name(f"{base_name}_{suffix}")

                new_ws.Name = target_name
                existing.add(target_name)

                sizing_block = sizing_all.get(comp_code) or {}
                write_component_sheet(new_ws, family_code, sole_type, comp_code, comp, sizing_block, warnings)

        app.DisplayAlerts = False
        ws_inv_template.Delete()
        app.DisplayAlerts = True

        wb.SaveAs(str(out_path), FileFormat=51)
    finally:
        wb.Close(SaveChanges=False)

    return str(out_path), warnings


def export_all_families(json_path, template_path, output_dir, max_families=None):
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))
    families = data.get("families") or {}
    if not families:
        raise ValueError("No families found in JSON.")

    
    pythoncom.CoInitialize()
    
    try:
        app = win32.gencache.EnsureDispatch("Excel.Application")
        excel_turbo_on(app)

        written_files = []
        all_warnings = []

        try:
            for i, (family_code, fam) in enumerate(families.items(), start=1):
                if max_families is not None and i > max_families:
                    break

                out_path, warnings = export_one_family(
                    template_path=template_path,
                    family_code=family_code,
                    fam=fam,
                    output_dir=output_dir,
                    app=app,
                )
                written_files.append(out_path)
                all_warnings.extend(warnings)
        finally:
            excel_turbo_off(app)
            app.Quit()
    finally:
        pythoncom.CoUninitialize()

    return written_files, all_warnings


# ----------------------------
# Run export
# ----------------------------
JSON_PATH = "../data/mold_data.json"
TEMPLATE_PATH = "../docs/tenplates/MoldFamily_(Template).xlsx"
OUTPUT_DIR = "../data/output"
MAX_FAMILIES = 4  # e.g., 3 for quick test

files, warnings = export_all_families(
    json_path=JSON_PATH,
    template_path=TEMPLATE_PATH,
    output_dir=OUTPUT_DIR,
    max_families=MAX_FAMILIES,
)

print(f"Exported {len(files)} workbook(s) to {OUTPUT_DIR}")
for path in files[:10]:
    print(" -", path)
if len(files) > 10:
    print(f" ... and {len(files)-10} more")

print(f"Warnings: {len(warnings)}")
for w in warnings[:30]:
    print(" !", w)
if len(warnings) > 30:
    print(f" ... and {len(warnings)-30} more")

Exported 4 workbook(s) to ../data/output
 - D:\Project\osms-mold-master-data\data\output\SA-1318.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2017.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2017-E.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2253.xlsx
Warnings: 0
